In [1]:
import spacy
import numpy as np
import pandas as pd
from spacy.training import offsets_to_biluo_tags
import torch
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from keras.preprocessing.sequence import pad_sequences
from pytorch_pretrained_bert import BertTokenizer, BertConfig
from pytorch_pretrained_bert import BertForTokenClassification
from seqeval.metrics import classification_report, accuracy_score, f1_score

In [2]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")
# Adding '\n' to the default spacy tokenizer
prefixes = ('\n', ) + tuple(nlp.Defaults.prefixes)
prefix_regex = spacy.util.compile_prefix_regex(prefixes)
nlp.tokenizer.prefix_search = prefix_regex.search

In [3]:
# Personal Custom Tags Dictionary
entity_dict = {
    'Name': 'NAME', 
    'College Name': 'CLG',
    'Degree': 'DEG',
    'Graduation Year': 'GRADYEAR',
    'Years of Experience': 'YOE',
    'Companies worked at': 'COMPANY',
    'Designation': 'DESIG',
    'Skills': 'SKILLS',
    'Location': 'LOC',
    'Email Address': 'EMAIL'
}

In [71]:
# Define tag2idx mapping (same as in training)
{'I-DESIG': 0,
 'I-COMPANY': 1,
 'B-COMPANY': 2,
 'L-LOC': 3,
 'U-DESIG': 4,
 'I-DEG': 5,
 'L-SKILLS': 6,
 'I-CLG': 7,
 'U-DEG': 8,
 'X': 9,
 'I-LOC': 10,
 'L-DEG': 11,
 'I-EMAIL': 12,
 'O': 13,
 'I-YOE': 14,
 'B-NAME': 15,
 '[CLS]': 16,
 'L-GRADYEAR': 17,
 'I-GRADYEAR': 18,
 'U-SKILLS': 19,
 'I-SKILLS': 20,
 'L-DESIG': 21,
 'L-YOE': 22,
 'U-YOE': 23,
 'B-EMAIL': 24,
 'L-NAME': 25,
 '[SEP]': 26,
 'B-GRADYEAR': 27,
 'L-COMPANY': 28,
 'L-EMAIL': 29,
 'B-CLG': 30,
 'I-NAME': 31,
 'U-CLG': 32,
 'L-CLG': 33,
 'U-GRADYEAR': 34,
 'B-LOC': 35,
 'U-EMAIL': 36,
 'U-COMPANY': 37,
 'B-DESIG': 38,
 'B-DEG': 39,
 'U-LOC': 40,
 'B-SKILLS': 41,
 'B-YOE': 42}
idx2tag = {tag2idx[key]: key for key in tag2idx.keys()}
print("tag2idx keys:", list(tag2idx.keys()))
print("Number of tags:", len(tag2idx))

tag2idx keys: ['U-DEG', 'I-DESIG', 'L-SKILLS', 'B-CLG', 'O', 'I-LOC', 'L-YOE', 'L-GRADYEAR', 'B-SKILLS', 'U-GRADYEAR', 'X', 'I-YOE', 'U-SKILLS', 'B-NAME', 'L-DESIG', 'U-COMPANY', 'U-CLG', 'I-GRADYEAR', 'I-NAME', 'B-DEG', 'L-LOC', 'U-NAME', 'B-LOC', 'I-DEG', 'B-DESIG', 'I-SKILLS', 'U-DESIG', 'L-CLG', 'L-EMAIL', 'L-DEG', 'I-CLG', 'B-EMAIL', 'L-NAME', 'U-EMAIL', 'L-COMPANY', 'U-LOC', '[CLS]', 'B-YOE', '[SEP]', 'U-YOE', 'I-EMAIL', 'B-GRADYEAR', 'I-COMPANY', 'B-COMPANY']
Number of tags: 44


In [43]:
# Function to merge overlapping intervals
def mergeIntervals(intervals):
    sorted_by_lower_bound = sorted(intervals, key=lambda tup: tup[0])
    merged = []
    for higher in sorted_by_lower_bound:
        if not merged:
            merged.append(higher)
        else:
            lower = merged[-1]
            if higher[0] <= lower[1]:
                if lower[2] is higher[2]:
                    upper_bound = max(lower[1], higher[1])
                    merged[-1] = (lower[0], upper_bound, lower[2])
                else:
                    if lower[1] > higher[1]:
                        merged[-1] = lower
                    else:
                        merged[-1] = (lower[0], higher[1], higher[2])
            else:
                merged.append(higher)
    return merged

In [44]:
# Extract entities from annotations
def get_entities(df):
    entities = []
    for i in range(len(df)):
        entity = []
        if df['annotation'][i] is not None:
            for annot in df['annotation'][i]:
                try:
                    ent = entity_dict[annot['label'][0]]
                    start = annot['points'][0]['start']
                    end = annot['points'][0]['end'] + 1
                    entity.append((start, end, ent))
                except:
                    pass
            entity = mergeIntervals(entity)
            entities.append(entity)
        else:
            entities.append([])
    return entities

In [45]:
# Prepare test data
def get_test_data(df):
    tags = []
    sentences = []
    for i in range(len(df)):
        text = df['content'][i]
        entities = df['entities'][i]
        doc = nlp(text)
        tag = offsets_to_biluo_tags(doc, entities)
        tmp = pd.DataFrame([list(doc), tag]).T
        loc = []
        for i in range(len(tmp)):
            if tmp[0][i].text == '.' and tmp[1][i] == 'O':
                loc.append(i)
        loc.append(len(doc))
        last = 0
        data = []
        for pos in loc:
            data.append([list(doc)[last:pos], tag[last:pos]])
            last = pos
        for d in data:
            tag = ['O' if t == '-' else t for t in d[1]]
            if len(set(tag)) > 1:
                sentences.append(d[0])
                tags.append(tag)
    return sentences, tags

In [46]:
# Tokenize test data
def get_tokenized_test_data(sentences, tags):
    tokenized_texts = []
    word_piece_labels = []
    for word_list, label in zip(sentences, tags):
        temp_lable = ['[CLS]']
        temp_token = ['[CLS]']
        for word, lab in zip(word_list, label):
            token_list = tokenizer.tokenize(word.text)
            for m, token in enumerate(token_list):
                temp_token.append(token)
                if m == 0:
                    temp_lable.append(lab)
                else:
                    temp_lable.append('X')
        temp_lable.append('[SEP]')
        temp_token.append('[SEP]')
        tokenized_texts.append(temp_token)
        word_piece_labels.append(temp_lable)
    return tokenized_texts, word_piece_labels

In [47]:
# Convert predictions to original format
def convert_predictions(predictions, word_piece_labels, tokenized_texts):
    pred_tags = []
    true_tags = []
    for i, pred in enumerate(predictions):
        pred_tag = []
        true_tag = []
        for j, p in enumerate(pred):
            if word_piece_labels[i][j] != 'X' and word_piece_labels[i][j] != '[CLS]' and word_piece_labels[i][j] != '[SEP]':
                pred_tag.append(idx2tag[p])
                true_tag.append(word_piece_labels[i][j])
        pred_tags.append(pred_tag)
        true_tags.append(true_tag)
    return pred_tags, true_tags

In [48]:
print("Loading test data...")
# Load test data
test_df = pd.read_json('./test.json')

Loading test data...


In [49]:
# Drop unnecessary columns
if 'extras' in test_df.columns:
    test_df = test_df.drop(['extras'], axis=1)
if 'metadata' in test_df.columns:
    test_df = test_df.drop(['metadata'], axis=1)

In [50]:
test_df.head()

,content,annotation
0,Vijay Shinde\nasst. sales manager in M - S PLA...,"[{'label': ['Email Address'], 'points': [{'sta..."
1,"Jaykumar Shah\nSales Manager\n\nMumbai, Mahara...","[{'label': ['Skills'], 'points': [{'start': 38..."
2,Raktim Podder\n6+ Exp in banking operations an...,"[{'label': ['Skills'], 'points': [{'start': 88..."
3,Bhaskar Gupta\nBusiness Management professiona...,"[{'label': ['Companies worked at'], 'points': ..."
4,Rohit Bijlani\nJAVA Developer/Senior Systems E...,"[{'label': ['Graduation Year'], 'points': [{'s..."


In [51]:
# Extract entities
test_df['entities'] = get_entities(test_df)
test_df['entities'].head()

0    [(0, 12, NAME), (13, 32, DESIG), (40, 67, COMP...
1    [(0, 13, NAME), (14, 27, DESIG), (29, 35, LOC)...
2    [(0, 13, NAME), (14, 17, YOE), (66, 70, LOC), ...
3    [(0, 13, NAME), (14, 46, DESIG), (57, 79, YOE)...
4    [(0, 13, NAME), (14, 52, DESIG), (55, 70, COMP...
Name: entities, dtype: object

In [52]:
# Prepare test data
test_sentences, test_tags = get_test_data(test_df)
print(f"Processed {len(test_sentences)} test sentences")

C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Vijay Shinde
asst. sales manager in M - S PLANET T..." with entities "[(0, 12, 'NAME'), (13, 32, 'DESIG'), (40, 67, 'COM...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Jaykumar Shah
Sales Manager

Mumbai, Maharashtra -..." with entities "[(0, 13, 'NAME'), (14, 27, 'DESIG'), (29, 35, 'LOC...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserW

Processed 547 test sentences


C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Raktim Podder
6+ Exp in banking operations and cre..." with entities "[(0, 13, 'NAME'), (14, 17, 'YOE'), (66, 70, 'LOC')...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\spacy\training\iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Navas Koya
Test Engineer

Mangalore, Karnataka - E..." with entities "[(0, 10, 'NAME'), (11, 25, 'DESIG'), (26, 35, 'LOC...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


In [53]:
test_df.head()

,content,annotation,entities
0,Vijay Shinde\nasst. sales manager in M - S PLA...,"[{'label': ['Email Address'], 'points': [{'sta...","[(0, 12, NAME), (13, 32, DESIG), (40, 67, COMP..."
1,"Jaykumar Shah\nSales Manager\n\nMumbai, Mahara...","[{'label': ['Skills'], 'points': [{'start': 38...","[(0, 13, NAME), (14, 27, DESIG), (29, 35, LOC)..."
2,Raktim Podder\n6+ Exp in banking operations an...,"[{'label': ['Skills'], 'points': [{'start': 88...","[(0, 13, NAME), (14, 17, YOE), (66, 70, LOC), ..."
3,Bhaskar Gupta\nBusiness Management professiona...,"[{'label': ['Companies worked at'], 'points': ...","[(0, 13, NAME), (14, 46, DESIG), (57, 79, YOE)..."
4,Rohit Bijlani\nJAVA Developer/Senior Systems E...,"[{'label': ['Graduation Year'], 'points': [{'s...","[(0, 13, NAME), (14, 52, DESIG), (55, 70, COMP..."


In [54]:
# Tokenize test data
tokenized_texts, word_piece_labels = get_tokenized_test_data(test_sentences, test_tags)
print(len(tokenized_texts))

547


In [55]:
# Convert to input format
MAX_LEN = 256
input_ids = pad_sequences([tokenizer.convert_tokens_to_ids(txt) for txt in tokenized_texts],
                          maxlen=MAX_LEN, dtype="long", truncating="post", padding="post")
tags = pad_sequences([[tag2idx.get(l) for l in lab] for lab in word_piece_labels], 
                     maxlen=MAX_LEN, value=tag2idx["O"], padding="post", 
                     dtype="long", truncating="post")
attention_masks = [[float(i>0) for i in ii] for ii in input_ids]

Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (1912 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (1132 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (599 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (920 > 512). Running this sequence through BERT will result in indexing errors
Token indices sequence length is longer than the specified maximum  sequence length for this BERT model (977 > 512). Running this sequence through BERT will result in indexing errors


In [56]:
# Convert to PyTorch tensors
test_inputs = torch.tensor(input_ids)
test_tags = torch.tensor(tags)
test_masks = torch.tensor(attention_masks)

In [57]:
# Create DataLoader
test_data = TensorDataset(test_inputs, test_masks, test_tags)
test_sampler = SequentialSampler(test_data)
test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=8)

In [72]:
# Alternative approach - load model then resize the classifier
print("Loading model from fifth_epoch.pt...")
model = BertForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=43  # Match the saved model first
)
model.load_state_dict(torch.load("fifth_epoch.pt"))
# Then resize the classifier to the new size
model.classifier = torch.nn.Linear(model.config.hidden_size, len(tag2idx))
model.config.num_labels = len(tag2idx)
print(f"Model loaded and classifier resized to {len(tag2idx)} labels")
print("Loaded classifier weights (pre-resize):")
print(model.classifier.weight[:2]) 

Loading model from fifth_epoch.pt...
Model loaded and classifier resized to 44 labels
Loaded classifier weights (pre-resize):
tensor([[-0.0062,  0.0234,  0.0238,  ...,  0.0032,  0.0075,  0.0172],
        [ 0.0066, -0.0257, -0.0121,  ...,  0.0265,  0.0169, -0.0073]],
       grad_fn=<SliceBackward0>)


In [60]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
# Evaluation mode
model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): BertLayerNorm()
              (dropout): Dropout(p=0.1, inplace=False

In [65]:
# Alternative simplified test function
def test_model_simple():
    model.eval()
    all_predictions = []
    all_true_labels = []
    print("Starting evaluation...")
    with torch.no_grad():
        for batch in test_dataloader:
            batch = tuple(t.to(device) for t in batch)
            b_input_ids, b_input_mask, b_labels = batch
            # Forward pass
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
            # Print output type and shape for debugging
            print(f"Output type: {type(outputs)}")
            if isinstance(outputs, tuple):
                print(f"Output[0] shape: {outputs[0].shape}")
            else:
                print(f"Output shape: {outputs.shape}")
            # Extract logits based on output type
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs
            # Get predictions - handle different shapes
            if len(logits.shape) == 3:
                preds = torch.argmax(logits, dim=2)
            else:
                # Reshape logits to match expected format
                batch_size = b_input_ids.size(0)
                seq_length = b_input_ids.size(1)
                num_labels = logits.size(-1)
                reshaped_logits = logits.view(batch_size, seq_length, num_labels)
                preds = torch.argmax(reshaped_logits, dim=2)
            # Convert to numpy for easier processing
            predictions = preds.detach().cpu().numpy()
            labels = b_labels.detach().cpu().numpy()
            # Store predictions and labels
            all_predictions.append(predictions)
            all_true_labels.append(labels)
    # Concatenate all batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_true_labels = np.concatenate(all_true_labels, axis=0)
    # Convert to tags and calculate metrics
    # (You'll need to adapt this part based on your convert_predictions function)
    print("Evaluation complete!")
    return all_predictions, all_true_labels

In [67]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

In [68]:
def convert_predictions_to_tags(preds, labels, idx2tag, ignore_label=-100):
    pred_tags = []
    true_tags = []
    for pred_seq, label_seq in zip(preds, labels):
        for p, l in zip(pred_seq, label_seq):
            if l == ignore_label:
                continue  # skip padding or special tokens
            pred_tags.append(idx2tag[p])
            true_tags.append(idx2tag[l])
    return pred_tags, true_tags

In [69]:
def evaluate_model():
    print("\nRunning evaluation...")
    preds, labels = test_model_simple()
    pred_tags, true_tags = convert_predictions_to_tags(preds, labels, idx2tag)
    print("\nMetrics:")
    print("F1 Score:", f1_score([true_tags], [pred_tags]))
    print("Precision:", precision_score([true_tags], [pred_tags]))
    print("Recall:", recall_score([true_tags], [pred_tags]))
    print("\nClassification Report:")
    print(classification_report([true_tags], [pred_tags]))

In [70]:
if __name__ == "__main__":
    tokenizer = BertTokenizer.from_pretrained('bert-base-cased', do_lower_case=False)
    evaluate_model()


Running evaluation...
Starting evaluation...
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torch.Size([8, 256, 44])
Output type: <class 'torch.Tensor'>
Output shape: torc

C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: [CLS] seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L-NAME seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: X seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L-DESIG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\user\anaconda3\envs\tff_en\lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: L-COMPANY seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:

F1 Score: 0.0036241045090602613
Precision: 0.0018999982325597837
Recall: 0.03914785142024763

Classification Report:
              precision    recall  f1-score   support

         CLG       0.00      0.16      0.00       231
        CLS]       0.00      0.00      0.00       547
     COMPANY       0.05      0.09      0.07       729
         DEG       0.00      0.01      0.00       176
       DESIG       0.00      0.02      0.00       437
       EMAIL       0.00      0.01      0.00        96
    GRADYEAR       0.00      0.00      0.00        72
         LOC       0.00      0.02      0.00       293
        NAME       0.00      0.19      0.00       218
        SEP]       0.00      0.00      0.00       508
      SKILLS       0.00      0.10      0.00       502
         YOE       0.00      0.00      0.00        57
           _       0.00      0.00      0.00      1626

   micro avg       0.00      0.04      0.00      5492
   macro avg       0.00      0.05      0.01      5492
weighted avg     